# Build an hourly NYC weather CSV from NOAA

This notebook downloads NOAA **Global Historical Climatology Network hourly (GHCNh)** observations for **NY City Central Park** and creates a CSV that can be merged with the NYC motor-vehicle-crash file using `crash_date` and `hour`.

**Date range used:** January 1, 2026 through July 1, 2026 (both dates included). This matches the 2026 crash-data period you have been working with. Change the dates in the settings cell if needed.

**Station:** NY CITY CENTRAL PARK, NY US (`USW00094728`), at approximately 40.77898, -73.96925. This is a representative station in the city, not a separate nearest station for each crash location.

**Important time detail:** NOAA's GHCNh observation timestamps are UTC. The notebook converts them to `America/New_York` before making `crash_date` and `hour`. That matches local New York clock time, including the switch from EST to EDT. It assumes your crash `hour` column is also local New York clock time (as recorded on the crash report), rather than fixed UTC-5 all year.

In [ ]:
import io
import requests
import pandas as pd
import numpy as np
from IPython.display import display


## 1. Settings

The end date is inclusive. The output will contain one row for every calendar date/hour combination from January 1 through July 1. The spring daylight-saving transition has no real 2:00 a.m. local hour; that row will remain in the file with missing weather values so the date/hour grid stays consistent.

In [ ]:
YEAR = 2026
START_DATE = "2026-01-01"
END_DATE = "2026-07-01"  # inclusive
TIME_ZONE = "America/New_York"

# NOAA GHCNh station: NY CITY CENTRAL PARK, NY US
STATION_ID = "USW00094728"
STATION_NAME = "NY CITY CENTRAL PARK, NY US"
STATION_LATITUDE = 40.77898
STATION_LONGITUDE = -73.96925

OUTPUT_FILE = "noaa_nyc_hourly_weather_2026.csv"

# NOAA GHCNh station-by-year pipe-separated file.
NOAA_URL = (
    "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/"
    f"hourly/access/by-year/{YEAR}/psv/GHCNh_{STATION_ID}_{YEAR}.psv"
)

start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE)
if start_date.year != YEAR or end_date.year != YEAR:
    raise ValueError("YEAR must match the year in START_DATE and END_DATE.")
if end_date < start_date:
    raise ValueError("END_DATE must be on or after START_DATE.")

print("Station:", STATION_NAME)
print("Station ID:", STATION_ID)
print("Dates:", START_DATE, "through", END_DATE, "inclusive")
print("NOAA file:", NOAA_URL)


## 2. Download the NOAA station file

NOAA provides GHCNh in a pipe-separated (`|`) format. This downloads only the selected station's file for the selected year, rather than the entire global dataset.

In [ ]:
response = requests.get(
    NOAA_URL,
    timeout=120,
    headers={"User-Agent": "NYC-crash-weather-analysis/1.0"}
)

if response.status_code == 404:
    raise FileNotFoundError(
        "NOAA returned 404 for this station/year file. Check that the station ID "
        "USW00094728 appears in NOAA's current GHCNh station list and that the "
        "2026 station file has been published. The file URL is printed above."
    )

response.raise_for_status()

weather_raw = pd.read_csv(
    io.BytesIO(response.content),
    sep="|",
    low_memory=False
)
weather_raw.columns = weather_raw.columns.str.strip()

print(f"Downloaded {len(weather_raw):,} NOAA observation rows.")
print(f"Number of source columns: {len(weather_raw.columns)}")
print("First columns:", weather_raw.columns[:12].tolist())


## 3. Check the station and observation timestamps

The GHCNh file contains a `DATE` timestamp in UTC. We convert that timestamp to New York local time before filtering the requested dates and extracting the local date/hour used for merging.

In [ ]:
def find_column(column_name):
    """Find a column without depending on capitalization."""
    matches = [c for c in weather_raw.columns if c.strip().casefold() == column_name.casefold()]
    return matches[0] if matches else None


date_column = find_column("DATE")
if date_column is None:
    raise KeyError(
        "Could not find the DATE column in the NOAA file. "
        "Inspect weather_raw.columns to check the file format."
    )

weather_raw["observation_time_utc"] = pd.to_datetime(
    weather_raw[date_column],
    errors="coerce",
    utc=True
)
weather_raw = weather_raw.dropna(subset=["observation_time_utc"]).copy()

# Use New York local midnight at the beginning and the day after END_DATE.
start_local = pd.Timestamp(f"{START_DATE} 00:00:00", tz=TIME_ZONE)
end_local_exclusive = pd.Timestamp(f"{END_DATE} 00:00:00", tz=TIME_ZONE) + pd.DateOffset(days=1)
start_utc = start_local.tz_convert("UTC")
end_utc = end_local_exclusive.tz_convert("UTC")

weather_raw = weather_raw.loc[
    (weather_raw["observation_time_utc"] >= start_utc)
    & (weather_raw["observation_time_utc"] < end_utc)
].copy()

if weather_raw.empty:
    raise ValueError(
        "No observations were found in the requested date range. "
        "Check the station ID, date range, and NOAA file contents."
    )

weather_raw["observation_time_local"] = weather_raw["observation_time_utc"].dt.tz_convert(TIME_ZONE)

print("First UTC observation:", weather_raw["observation_time_utc"].min())
print("Last UTC observation:", weather_raw["observation_time_utc"].max())
print("First New York observation:", weather_raw["observation_time_local"].min())
print("Last New York observation:", weather_raw["observation_time_local"].max())

# Show station metadata from the source file when those fields are available.
for possible_name in ["Station_ID", "Station_name", "Latitude", "Longitude", "Elevation"]:
    col = find_column(possible_name)
    if col is not None:
        print(f"{possible_name}:", weather_raw[col].dropna().astype(str).unique()[:3])


## 4. Keep useful weather variables

The output uses NOAA's reported units:

- Temperature and dew point: degrees Celsius
- Precipitation: millimeters
- Wind speed and gust: meters per second
- Wind direction: degrees clockwise from true north
- Visibility: kilometers
- Pressure: hectopascals (hPa)
- Relative humidity: percent

Weather-condition fields are kept as NOAA codes where available. The notebook also keeps selected measurement/quality-code fields so you can inspect trace precipitation and source quality flags instead of silently discarding them.

In [ ]:
# NOAA source variable -> output column. Some variables may not be present for every station.
variable_specs = [
    ("temperature", "temperature_c", "numeric"),
    ("dew_point_temperature", "dew_point_c", "numeric"),
    ("relative_humidity", "relative_humidity_pct", "numeric"),
    ("precipitation", "precipitation_mm", "numeric"),
    ("wind_speed", "wind_speed_m_s", "numeric"),
    ("wind_direction", "wind_direction_deg", "numeric"),
    ("wind_gust", "wind_gust_m_s", "numeric"),
    ("visibility", "visibility_km", "numeric"),
    ("sea_level_pressure", "sea_level_pressure_hpa", "numeric"),
    ("station_level_pressure", "station_pressure_hpa", "numeric"),
    ("altimeter", "altimeter_hpa", "numeric"),
    ("ceiling_height", "ceiling_height_m", "numeric"),
    ("snow_depth", "snow_depth_mm", "numeric"),
    ("sky_condition", "sky_condition_code", "code"),
    ("pres_wx_AU1", "automated_weather_code_1", "code"),
    ("pres_wx_AU2", "automated_weather_code_2", "code"),
    ("pres_wx_AU3", "automated_weather_code_3", "code"),
    ("pres_wx_MW1", "manual_weather_code_1", "code"),
    ("pres_wx_MW2", "manual_weather_code_2", "code"),
    ("pres_wx_MW3", "manual_weather_code_3", "code"),
]

# Match NOAA headers case-insensitively.
source_columns = {c.casefold(): c for c in weather_raw.columns}
selected_columns = {}
numeric_output_columns = []

for source_name, output_name, data_type in variable_specs:
    source_column = source_columns.get(source_name.casefold())
    if source_column is not None:
        selected_columns[source_column] = output_name
        if data_type == "numeric":
            numeric_output_columns.append(output_name)

if not selected_columns:
    raise KeyError("None of the expected weather variables were found in the NOAA file.")

# Preserve selected metadata fields when present.
metadata_specs = [
    ("precipitation_Measurement_Code", "precipitation_measurement_code"),
    ("temperature_Quality_Code", "temperature_qc"),
    ("dew_point_temperature_Quality_Code", "dew_point_qc"),
    ("relative_humidity_Quality_Code", "relative_humidity_qc"),
    ("precipitation_Quality_Code", "precipitation_qc"),
    ("wind_speed_Quality_Code", "wind_speed_qc"),
    ("visibility_Quality_Code", "visibility_qc"),
]

for source_name, output_name in metadata_specs:
    source_column = source_columns.get(source_name.casefold())
    if source_column is not None:
        selected_columns[source_column] = output_name

print("Weather fields available:")
for source_column, output_column in selected_columns.items():
    print(f"  {source_column} -> {output_column}")


In [ ]:
# Build a smaller table with local date/hour merge keys.
keep_columns = ["observation_time_local"] + list(selected_columns.keys())
observations = weather_raw[keep_columns].copy()
observations = observations.rename(columns=selected_columns)

# Convert measured weather fields to numbers. NOAA missing-value sentinels are set to NaN.
missing_sentinels = [-9999, -999.9, 9999, 999.9]
for column in numeric_output_columns:
    observations[column] = pd.to_numeric(observations[column], errors="coerce")
    observations[column] = observations[column].replace(missing_sentinels, np.nan)

# NOAA timestamps are converted to local New York time above.
observations["crash_date"] = observations["observation_time_local"].dt.strftime("%Y-%m-%d")
observations["hour"] = observations["observation_time_local"].dt.hour.astype("int64")
observations["local_observation_time"] = observations["observation_time_local"].dt.strftime("%Y-%m-%d %H:%M:%S")
observations["weather_observation_count"] = 1

# A trace flag is useful because a trace amount may not have a numeric precipitation value.
precip_code_col = "precipitation_measurement_code"
if precip_code_col in observations.columns:
    observations["precipitation_trace"] = (
        observations[precip_code_col].astype("string").str.strip().str.upper().eq("T")
    )

observations = observations.sort_values("observation_time_local")

# Use the final available report/value in each local clock hour.
# NOAA's GHCNh documentation specifically says to use the last report in an hour
# when extracting hourly precipitation from stations with multiple METAR reports.
aggregation = {
    column: "last"
    for column in observations.columns
    if column not in ["observation_time_local", "crash_date", "hour", "weather_observation_count"]
}
aggregation["local_observation_time"] = "last"
aggregation["weather_observation_count"] = "sum"
if "precipitation_mm" in aggregation:
    aggregation["precipitation_mm"] = lambda values: values.iloc[-1]

weather_by_hour = (
    observations
    .groupby(["crash_date", "hour"], as_index=False)
    .agg(aggregation)
)

print(f"Observed local date/hour combinations: {len(weather_by_hour):,}")
display(weather_by_hour.head())


## 5. Create a complete local date/hour grid

The grid includes all 24 hour values for each calendar date. If NOAA did not report an observation for a date/hour, the weather columns stay missing and `weather_observation_count` is 0. On March 8, 2026, the local 2 a.m. clock hour does not occur because clocks move forward; its row is therefore expected to have missing weather values.

In [ ]:
date_values = pd.date_range(START_DATE, END_DATE, freq="D").strftime("%Y-%m-%d")
full_grid = pd.MultiIndex.from_product(
    [date_values, range(24)],
    names=["crash_date", "hour"]
).to_frame(index=False)

weather = full_grid.merge(
    weather_by_hour,
    on=["crash_date", "hour"],
    how="left",
    validate="one_to_one"
)

weather["weather_observation_count"] = (
    weather["weather_observation_count"].fillna(0).astype("int64")
)
weather["station_id"] = STATION_ID
weather["station_name"] = STATION_NAME
weather["station_latitude"] = STATION_LATITUDE
weather["station_longitude"] = STATION_LONGITUDE

# Put the merge keys first, followed by station and weather information.
first_columns = [
    "crash_date", "hour", "local_observation_time", "weather_observation_count",
    "station_id", "station_name", "station_latitude", "station_longitude"
]
remaining_columns = [c for c in weather.columns if c not in first_columns]
weather = weather[first_columns + remaining_columns].sort_values(["crash_date", "hour"]).reset_index(drop=True)

print("Rows in output:", f"{len(weather):,}")
print("Expected rows:", f"{len(date_values) * 24:,}")
print("Unique date/hour keys:", weather[["crash_date", "hour"]].drop_duplicates().shape[0])
print("Duplicate merge keys:", weather.duplicated(["crash_date", "hour"]).sum())


## 6. Check missing weather values

Missing values are not automatically filled. A blank means that the selected NOAA station did not provide that measurement for that local hour. The observation count helps distinguish hours with no station report from hours where a report exists but a particular variable is missing.

In [ ]:
weather_columns = [
    c for c in weather.columns
    if c not in [
        "crash_date", "hour", "local_observation_time", "weather_observation_count",
        "station_id", "station_name", "station_latitude", "station_longitude"
    ]
]

missing_summary = pd.DataFrame({
    "missing_count": weather[weather_columns].isna().sum(),
    "missing_percent": (weather[weather_columns].isna().mean() * 100).round(1)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)
print("Hours with at least one NOAA observation:", int((weather["weather_observation_count"] > 0).sum()))
print("Hours with no NOAA observations:", int((weather["weather_observation_count"] == 0).sum()))

display(weather.head(12))


## 7. Save the weather CSV

The CSV is saved in the notebook's current working folder. If you are running this in Google Colab, the final cell also starts a download to your computer.

In [ ]:
weather.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")
from pathlib import Path
print(f"File size: {Path(OUTPUT_FILE).stat().st_size / 1024:.1f} KB")

# In Google Colab, download the CSV to your computer.
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not running in Google Colab. The CSV is saved in the current working folder.")


## 8. Merge with the NYC crash dataset later

Your crash notebook already uses `crash_date` and `hour`. Convert both datasets to matching key types before merging. This is a **left merge**, so every crash is kept even when the weather station has no observation for that hour.

In [ ]:
# Example for when you are ready to merge:
#
# crashes["crash_date"] = pd.to_datetime(crashes["crash_date"]).dt.strftime("%Y-%m-%d")
# crashes["hour"] = pd.to_numeric(crashes["hour"], errors="coerce").astype("Int64")
#
# weather["crash_date"] = pd.to_datetime(weather["crash_date"]).dt.strftime("%Y-%m-%d")
# weather["hour"] = pd.to_numeric(weather["hour"], errors="coerce").astype("Int64")
#
# crashes_with_weather = crashes.merge(
#     weather,
#     on=["crash_date", "hour"],
#     how="left",
#     validate="many_to_one"
# )
#
# print("Crash rows before merge:", len(crashes))
# print("Crash rows after merge:", len(crashes_with_weather))
# display(crashes_with_weather.head())


## NOAA source and citation

NOAA National Centers for Environmental Information (NCEI), **Global Historical Climatology Network hourly (GHCNh)**, Version 1. NOAA describes GHCNh as its newer hourly/synoptic station dataset replacing the older Integrated Surface Database. The GHCNh station files include UTC observation timestamps and pipe-separated weather variables.

- Dataset overview: https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly
- GHCNh documentation: https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf
- Central Park station record: https://www.ncei.noaa.gov/cdo-web/datasets/GHCND/stations/GHCND%3AUSW00094728/detail

Dataset citation: Menne, Matthew J.; Noone, Simon; Casey, Nancy W.; Dunn, Robert H.; McNeill, Shelley; Kantor, Diana; Thorne, Peter W.; Orcutt, Karen; Cunningham, Sam; Risavi, Nicholas. 2023. *Global Historical Climatology Network-Hourly (GHCNh).* NOAA National Centers for Environmental Information. https://doi.org/10.25921/jp3d-3v19.